# Lab 4: Wrangling Data with dplyr

<img src="ds4all-logo.png" alt="Data Science for All Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

Welcome to Lab 4 of *Data Science for All*!

Last week you read data into R and plotted it. This week you get a messy file into shape and ask it
questions with the **dplyr** verbs. These are the same skills Coding Assignment 2 asks for, practiced here on different data.

This lab will be due on **Thursday at 5 PM (Week 6)**. As always, collaborating on labs is encouraged,
but don't just share answers.

## Today's lab

In today's lab, you'll learn how to:

1. read a file whose values are not really separated by commas, and fix its column types;
2. find missing values that are hiding as zeros, and turn them into `NA`;
3. answer questions with `filter()`, `arrange()`, `mutate()`, `summarize()`, and `pull()`; and
4. summarize by group, then plot the result.

## Resources

For review:

- **Class notes:** [Grammar of Data Wrangling](https://gwu.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fsarahcburnett%2Fds4all-materials-f26-public&branch=main&urlpath=tree%2Fds4all-materials-f26-public%2Flecture-notebooks%2Ftidyverse%2FGrammar_of_Data_Wrangling.ipynb)
- **Reading:** [R for Data Science, Chapter 3: Data transformation](https://r4ds.hadley.nz/data-transform.html)

Helpful to keep open while you work: **dplyr cheat sheet** [PDF](https://rstudio.github.io/cheatsheets/data-transformation.pdf) or [web version](https://rstudio.github.io/cheatsheets/html/data-transformation.html)

Each dplyr verb does one job. When you are not sure which one you need, start here:

| If you want to... | Use |
|---|---|
| keep some rows | `filter()` |
| sort the rows | `arrange()` |
| keep some columns | `select()` |
| rename columns | `rename()` |
| add or change a column | `mutate()` |
| summarize many rows into one row | `summarize()` |
| do that for each group | `group_by()`, then `summarize()` |
| count the rows in each group | `count()` |
| take a column out as a vector, or a single number | `pull()` |

In [ ]:
# Initialize the autograder and JupyterHub helper
source("helpers.R")

library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)

options(repr.matrix.max.rows = 20)
options(repr.plot.width = 7, repr.plot.height = 4.5)

Last week we loaded all of `library(tidyverse)`. This week we load only the four packages that
Coding Assignment 2's autograder has, so anything that runs here will also run there.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 1. Reading a messy file

Our data this week: 406 cars from the 1970 to 1982 model years, with their fuel efficiency, engine
size, weight, and where they were made.

`cars.csv` comes from the [project datasets page](https://perso.telecom-paristech.fr/eagan/class/igr204/datasets)
for IGR 204, a data visualization course taught by James Eagan at Télécom Paris. The page collects real
datasets for student projects, many of them cleaned up by visualization researchers Petra Isenberg,
Pierre Dragicevic, and Yvonne Jansen.

The cars come from the Auto MPG dataset, a classic in statistics and machine learning: Quinlan, R. (1993). *Auto MPG* [Dataset]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5859H>. Quinlan used it to predict a car's fuel efficiency (MPG) from its other variables.

The file ends in `.csv`, so `read_csv()` looks like the obvious choice. Try it:

In [ ]:
cars_try <- read_csv("data/cars.csv", show_col_types = FALSE)

cars_try

That gave us a single column: each row came in as one long piece of text. When a file reads in wrong, look at the raw
text. `read_lines()` shows the first few lines exactly as they sit in the file:

In [ ]:
read_lines("data/cars.csv", n_max = 4)

The raw text shows two problems:

1. The values are separated by **semicolons**, not commas. (Remember 'csv' stands for comma-separated values). `read_delim()` reads a file with any
   separator. Tell it which one with `delim`.
2. The second line is not a car. It lists each column's type (`STRING`, `DOUBLE`, ...). Because that
   row is text, every column comes in as text.

The fix takes three steps: read with the right separator, drop the row of types with `slice(-1)`, and
let `type_convert()` guess each column's type again now that the text row is gone.
(`col_types = cols()` just stops it from printing a summary.)

---

**Question 1.1.** Read `data/cars.csv` into a tibble named `cars_raw` using those three steps. Fill in
the blanks.

In [ ]:
cars_raw <- read_delim("...", delim = "...", show_col_types = FALSE) |>
  slice(...) |>
  type_convert(col_types = cols())

cars_raw

In [ ]:
. <- ottr::check("tests/q1_1.R")

Before we clean anything, check the types. Every column except `Car` and `Origin` should now be `<dbl>`, a number.

In [ ]:
glimpse(cars_raw)

Here is what each column means:

| Column | Meaning |
|---|---|
| `Car` | make and model |
| `MPG` | fuel efficiency, in miles per gallon |
| `Cylinders` | number of cylinders in the engine |
| `Displacement` | engine size, in cubic inches |
| `Horsepower` | engine power |
| `Weight` | weight, in pounds |
| `Acceleration` | seconds to go from 0 to 60 mph |
| `Model` | model year, written as 70 for 1970 |
| `Origin` | where the car was made: US, Europe, or Japan |

Coding Assignment 2's `film.csv` has the same two problems, which is why its loading code uses `read_delim()`
and `slice(-1)`.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 2. Cleaning: missing values in disguise

The file reads in cleanly now, but that does not mean every value makes sense. Look at the cars
with an MPG of 0:

In [ ]:
cars_raw |> filter(MPG == 0)

Eight cars get 0 miles per gallon. A car that gets 0 MPG cannot drive anywhere. These are **missing
values** that somebody stored as `0`. `Horsepower` has the same problem:

In [ ]:
cars_raw |> filter(Horsepower == 0)

<div style="border-left: 4px solid #fdb515; background: #fffdf5; padding: 0.8em 1em; margin: 1em 0;">
<b>A zero is worse than a blank.</b>
<p>R cannot tell a fake 0 from a real one, so it averages it in like any other number, and every
summary comes out wrong without an error or a warning. Missing values stored as 0, 99, or -1 are one
of the most common problems in real data. Look for them before you summarize anything.</p>
</div>

R's value for "missing" is `NA`. `na_if(x, 0)` turns every 0 in `x` into `NA`:

In [ ]:
na_if(c(18, 0, 25), 0)

Besides the zeros, a few columns need work. `Model` holds the model year as a two-digit number, so `70` means 1970. A better name is
`Year`, and a better value is the full year. Years are whole numbers, so store them as integers with
`as.integer()`.

Two more columns have the wrong type. `Cylinders` only ever holds whole numbers, so it should be an
integer too. `Origin` came in as text (`<chr>`), but it is a category with three values, so it should be a
factor. Give it the levels in the order US, Europe, Japan, and every table and legend that uses `Origin`
will list them in that order.

A reminder from Lab 3: `rename()` takes `new_name = old_name`, and a column name with spaces, symbols,
or a leading number needs backticks. In Coding Assignment 2, you will meet a column called `NEI-PTER`. Without backticks, R reads that as `NEI` minus `PTER`.

---

**Question 2.1.** Clean `cars_raw` into a new tibble called `cars_clean`, in one pipeline:

1. rename `Model` to `Year`;
2. turn `Year` into the full year, stored as an integer (70 becomes 1970);
3. replace the 0s in `MPG` and in `Horsepower` with `NA`; and
4. store `Cylinders` as an integer, and `Origin` as a factor with the levels US, Europe, Japan.

Keep every car. Fill in the blanks.

In [ ]:
names(cars_raw)

In [ ]:
cars_clean <- cars_raw |>
  rename(Year = ...) |>
  mutate(
    Year       = as.integer(... + Year),
    MPG        = na_if(..., ...),
    Horsepower = ...,
    Cylinders  = ...,
    Origin     = ...
  )

cars_clean

In [ ]:
. <- ottr::check("tests/q2_1.R")

How much did those zeros matter? To find out, we'll average `MPG` with and without them. First, two
tools.

`mean()` needs numbers, not a table. `select()` keeps a column but hands back a **table**. `pull()` hands back the
**values** inside it:

In [ ]:
cars_clean |> select(MPG) |> head(3)   # a table with one column

In [ ]:
cars_clean |> pull(MPG) |> head(3)     # a vector of numbers

`mean()` has one more catch. A single `NA` makes the whole average `NA`, because R will not guess what
the missing value was. `na.rm = TRUE` tells it to leave the `NA`s out:

In [ ]:
mean(c(20, 30, NA))

In [ ]:
mean(c(20, 30, NA), na.rm = TRUE)

---

**Question 2.2.** Compute the average MPG twice. Store each one as a single number, not a table.

- `mpg_with_zeros`: the average of `MPG` in `cars_raw`, where the fake 0s are still in.
- `mpg_without_zeros`: the average of `MPG` in `cars_clean`, where they are now `NA`.

In [ ]:
mpg_with_zeros    <- NULL # YOUR CODE HERE
mpg_without_zeros <- NULL # YOUR CODE HERE

c(mpg_with_zeros, mpg_without_zeros)

In [ ]:
. <- ottr::check("tests/q2_2.R")

Eight fake zeros out of 406 cars moved the average by about half a mile per gallon, with no error and
no warning.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 3. Asking questions with dplyr verbs

Each question in this section takes one or two verbs. Before you write code, decide which rows you
need, in what order, and whether the answer is a table or a single number. The table at the top of the
notebook is there when you need it.

---

**Question 3.1.** Which cars are the most fuel efficient? Make a tibble named `best_mpg` with the 10
cars in `cars_clean` that have the highest MPG, best first.

*Hint:* `arrange()` sorts from low to high, and `desc()` flips it. `head()` keeps the first rows.

In [ ]:
best_mpg <- NULL # YOUR CODE HERE

best_mpg

In [ ]:
. <- ottr::check("tests/q3_1.R")

---

**Question 3.2.** Car magazines compare engines with a **power-to-weight ratio**: horsepower per 1,000
pounds of car. Add a column `power_to_weight` to `cars_clean`, computed as
$$
\text{power-to-weight} = \frac{\text{Horsepower}}{\text{Weight}} \times 1000,
$$

then sort from highest to lowest. Name the result `cars_power`. Write it
as one pipeline.

In [ ]:
cars_power <- NULL # YOUR CODE HERE

cars_power

In [ ]:
. <- ottr::check("tests/q3_2.R")

Look at the car at the top: a 225-horsepower station wagon that weighs 3,086 pounds. Every other car
in this data with 200 or more horsepower weighs over 4,000 pounds, so that weight is probably a typo in
the original data. A new column is a good way to catch values like this: when one number is wrong, a
ratio built from it stands out.

The next question asks how far each car is from the average, in either direction. Recall the function
`abs()`.

In [ ]:
abs(c(-12, 3, 12))

---

**Question 3.3.** How many cars in `cars_clean` have an MPG more than 10 away from the average MPG,
either above or below it? Store the count as a single number in `far_from_avg`. Fill in the blanks.

In [ ]:
far_from_avg <- cars_clean |>
  filter(abs(... - mean(..., na.rm = TRUE)) > ...) |>
  nrow()

far_from_avg

In [ ]:
. <- ottr::check("tests/q3_3.R")

The last question in this section asks for one number about one group of cars. `summarize()`
collapses a table into one row. The result is still a table, so `pull()` takes the number
out of it:

In [ ]:
cars_clean |>
  summarize(avg_weight = mean(Weight))

In [ ]:
cars_clean |>
  summarize(avg_weight = mean(Weight)) |>
  pull(avg_weight)

---

**Question 3.4.** What is the average MPG of the cars made in Japan? Store it as a single number in
`japan_mpg`.

*Hint:* keep the Japanese cars first, then summarize, then pull. If you are not sure how the origins
are spelled, `levels(cars_clean$Origin)` lists them.

In [ ]:
japan_mpg <- NULL # YOUR CODE HERE

japan_mpg

In [ ]:
. <- ottr::check("tests/q3_4.R")

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 4. Wrangle, then plot

Some questions need a summary before there is anything to plot. `group_by()` splits the table into
groups, and `summarize()` then works on each group separately, giving one row per group:

In [ ]:
cars_clean |>
  group_by(Origin) |>
  summarize(avg_weight = mean(Weight))

Three origins in, three rows out, listed in the order of the levels you gave `Origin` in Question 2.1.

Every plot in this section follows the labeling standards from Lab 3: a title that names the variables
or states the finding, axis labels in plain words with units, a legend title when there is a legend,
and a caption naming the source. For this data, use `caption = "Source: Auto MPG, UCI Machine Learning Repository"`.

---

**Question 4.1.** *Did cars get more fuel efficient over time?*

First make `mpg_by_year`: one row per model year, with the average MPG in a column called `avg_mpg`.
Then plot it as a line chart named `mpg_line`. Fill in the blanks.

The `scale_x_continuous()` line is done for you. Without it, `ggplot2` puts tick marks at in-between
values like 1972.5, which is not a model year.

In [ ]:
mpg_by_year <- cars_clean |>
  group_by(...) |>
  summarize(avg_mpg = mean(..., na.rm = TRUE))

mpg_line <- ggplot(mpg_by_year) +
  aes(x = ..., y = ...) +
  geom_...(linewidth = 1) +
  scale_x_continuous(breaks = seq(1970, 1982, by = 2)) +
  labs(
    title = "...",
    x = "...",
    y = "...",
    caption = "Source: Auto MPG, UCI Machine Learning Repository"
  ) +
  theme_minimal(base_size = 16)

mpg_line

Now read the line. Did the average MPG go up **every single year** from 1970 to 1982? Set
`rose_every_year` to `TRUE` or `FALSE`.

In [ ]:
rose_every_year <- NULL # YOUR CODE HERE

rose_every_year

In [ ]:
. <- ottr::check("tests/q4_1.R")

Gas prices spiked in the 1973 oil crisis. Look at what the line does for the models that came after
it. A jump right after a big event is a lead worth following, not proof that the event caused it.

---

**Question 4.2.** *Is there an association between a car's weight and its fuel efficiency?*

Make a scatter plot from `cars_clean` with `Weight` on the x-axis and `MPG` on the y-axis. Name it
`weight_scatter`. Use `alpha = 0.5`, and give it a title, axis labels in plain words with units, a
caption, and `theme_minimal(base_size = 16)`.

You will see a warning that 8 rows were removed. Those are the cars with a missing MPG. `ggplot2` cannot
place a point without a y value, so it leaves them out and tells you.

In [ ]:
weight_scatter <- NULL # YOUR CODE HERE

weight_scatter

Which statement describes `weight_scatter` best?

- `A.` Strong positive association: heavier cars get more miles per gallon.
- `B.` Strong negative association: heavier cars tend to get fewer miles per gallon, and the points follow that downward pattern closely.
- `C.` No association: the cloud has no shape, so weight tells you nothing about MPG.
- `D.` The plot has too many points to say anything about these two variables.

Set `weight_answer` to the letter, as a string.

In [ ]:
weight_answer <- NULL # YOUR CODE HERE

weight_answer

In [ ]:
. <- ottr::check("tests/q4_2.R")

---

**Question 4.3.** *Where were the cars made, and did that change over time?*

Make a bar chart from `cars_clean` with one bar per model year, filled by `Origin`. Use the geom that
counts rows for you. Name it `origin_bar`, and give it a title, axis labels, a
legend title, and a caption.

In [ ]:
origin_bar <- NULL # YOUR CODE HERE

origin_bar

Now compute the **share** of the 1980 models that were made in Japan, as a proportion between 0 and 1.
Store it as a single number in `japan_share_1980`.

*Hint:* `Origin == "Japan"` is `TRUE` or `FALSE` for each car, and the mean of a column of
`TRUE`/`FALSE` values is the share that are `TRUE`.

In [ ]:
japan_share_1980 <- NULL # YOUR CODE HERE

japan_share_1980

In [ ]:
. <- ottr::check("tests/q4_3.R")

The US made the most cars in every year except 1980. But a count and a share answer different
questions. In 1970, about 1 car in 17 came from Japan. By 1980, it was close to half. When groups differ
in size, compare shares, not just counts.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


You're done with Lab 4!

Here's what you can now do: read a file that is not what its extension says, find missing values
hiding as zeros, and answer questions with a verb or two: sort, filter, add a column, summarize, pull
out a single number, and summarize by group before plotting.

### Where this shows up in Coding Assignment 2

| In this lab | In Coding Assignment 2 |
|---|---|
| 1.1 `read_delim()` and the row of types | Part 3, loading `film.csv` |
| 2.1 `rename()`, `mutate()`, integers, and `NA` | Task 10, cleaning the films |
| 2.2 and 3.4 a single number with `pull()` | Tasks 2, 6, and 8 |
| 3.1 the top rows with `arrange(desc())` and `head()` | Task 1 |
| 3.2 a new column, then sort, in one pipeline | Task 3 |
| 3.3 `abs()` and `nrow()` | Task 7 |
| 4.1 a line chart over time | Task 4 |
| 4.2 a scatter plot for association | Task 9 |
| 4.3 a filled bar chart, and a share | Tasks 11 and 6 |

When you get stuck on the coding assignment, come back to the matching question here.

**Important submission information:** Be sure to run the tests and verify that they all pass, then
choose **Save Notebook** from the **File** menu, then Download the .ipynb. Then, go to your course's
Gradescope and submit the corresponding assignment. The name of this assignment is "Lab 04".

### Moral support of this data science course

Good work! You've earned a hug.

<img src="hug.jpg" alt="A boy hugging a cat" width="300"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
# Re-run all the autograder tests in the tests/ folder.
. <- sapply(list.files("tests", pattern = "\\.R$", full.names = TRUE), function(f) ottr::check(f))

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all
images/graphs appear in the output. Click Download in the bar at the top of the notebook. **Please save
before downloading!** Submit your final version to Gradescope.